#Read Me

1. load all darkcast data from github
2. `decay_profile(mX,alphaX,x)` returns dictionary containing total width and all branching ratios. **This is not vectorized**.
3. `get_decay_length(mX,alphaX,x)` returns a decay length using `decay_profile()`

# Initialize

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import InterpolatedUnivariateSpline

# constants
me = 0.511e-3 # GeV
mmu = 0.1057 # GeV
e = 0.303
c = 3e8 # m/s
mp = 0.9382720813

# gauge coupling charges
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}


#photoproduction model params CONVERGED
params = np.array([11.81336049,
                   3.92112205,
                   0.77877409,
                   0.68472406,
                   0.89693757,
                   0.80841249,
                   0.61509084,
                   0.69624785,
                   -4.21703614,
                   1.90289369,
                   8.81722149,
                   3.12168375,
                   1.1120275,
                   9.47975335,
                   0.92746299,
                   0.80066329,
                   0.80370305,
                   0.56739554,
                   1.74087675,
                   2.21138913,
                   5.52013592,
                   1.06419093,
                   0.99383077])


# format data
data_rho_pi_plus_pi_minus = [float(x) for x in open('rf.pi+_pi-.rho0.dat').read().split()[3:]]
data_rho_pi_plus_pi_minus_pi_plus_pi_minus = [float(x) for x in open('rf.pi+_pi-_pi+_pi-.rho0.dat').read().split()[3:]]
data_rho_pi_plus_pi_minus_pi0_pi0 = [float(x) for x in open('rf.pi+_pi-_pi0_pi0.rho0.dat').read().split()[3:]]
data_omega_pi_plus_pi_minus_pi0 = [float(x) for x in open('rf.pi+_pi-_pi0.omega.dat').read().split()[3:]]
data_phi_pi_plus_pi_minus_pi0 = [float(x) for x in open('rf.pi+_pi-_pi0.phi.dat').read().split()[3:]]
data_omega_pi0_gamma = [float(x) for x in open('rf.pi0_gamma.omega.dat').read().split()[3:]]
data_omega_phi_pi_plus_pi_minus_pi0 = [float(x) for x in open('rf.pi+_pi-_pi0.omega_phi.dat').read().split()[3:]]
data_phi_K_K = [float(x) for x in open('rf.K_K.phi.dat').read().split()[3:]]
data_phi_K_K_pi = [float(x) for x in open('rf.K_K_pi.phi.dat').read().split()[3:]]
data_omega_other = [float(x) for x in open('rf.other.omega.dat').read().split()[3:]]
data_phi_other = [float(x) for x in open('rf.other.phi.dat').read().split()[3:]]
data_rho_other = [float(x) for x in open('rf.other.rho0.dat').read().split()[3:]]

mass_data_rho_pi_plus_pi_minus = data_rho_pi_plus_pi_minus[0::2]
R_data_rho_pi_plus_pi_minus = data_rho_pi_plus_pi_minus[1::2]
mass_data_rho_pi_plus_pi_minus_pi_plus_pi_minus = data_rho_pi_plus_pi_minus_pi_plus_pi_minus[0::2]
R_data_rho_pi_plus_pi_minus_pi_plus_pi_minus = data_rho_pi_plus_pi_minus_pi_plus_pi_minus[1::2]
mass_data_rho_pi_plus_pi_minus_pi0_pi0 = data_rho_pi_plus_pi_minus_pi0_pi0[0::2]
R_data_rho_pi_plus_pi_minus_pi0_pi0 = data_rho_pi_plus_pi_minus_pi0_pi0[1::2]
mass_data_omega_pi_plus_pi_minus_pi0 = data_omega_pi_plus_pi_minus_pi0[0::2]
R_data_omega_pi_plus_pi_minus_pi0 = data_omega_pi_plus_pi_minus_pi0[1::2]
mass_data_phi_pi_plus_pi_minus_pi0 = data_phi_pi_plus_pi_minus_pi0[0::2]
R_data_phi_pi_plus_pi_minus_pi0 = data_phi_pi_plus_pi_minus_pi0[1::2]
mass_data_omega_pi0_gamma = data_omega_pi0_gamma[0::2]
R_data_omega_pi0_gamma = data_omega_pi0_gamma[1::2]
mass_data_omega_phi_pi_plus_pi_minus_pi0 = data_omega_phi_pi_plus_pi_minus_pi0[0::2]
R_data_omega_phi_pi_plus_pi_minus_pi0 = data_omega_phi_pi_plus_pi_minus_pi0[1::2]
mass_data_phi_K_K = data_phi_K_K[0::2]
R_data_phi_K_K = data_phi_K_K[1::2]
mass_data_phi_K_K_pi = data_phi_K_K_pi[0::2]
R_data_phi_K_K_pi = data_phi_K_K_pi[1::2]
mass_data_omega_other = data_omega_other[0::2]
R_data_omega_other = data_omega_other[1::2]
mass_data_phi_other = data_phi_other[0::2]
R_data_phi_other = data_phi_other[1::2]
mass_data_rho_other = data_rho_other[0::2]
R_data_rho_other = data_rho_other[1::2]

# interpolated splines
spline_rho_pi_plus_pi_minus = InterpolatedUnivariateSpline(
    mass_data_rho_pi_plus_pi_minus,
    R_data_rho_pi_plus_pi_minus,
    ext=1
)

spline_rho_pi_plus_pi_minus_pi_plus_pi_minus = InterpolatedUnivariateSpline(
    mass_data_rho_pi_plus_pi_minus_pi_plus_pi_minus,
    R_data_rho_pi_plus_pi_minus_pi_plus_pi_minus,
    ext=1
)

spline_rho_pi_plus_pi_minus_pi0_pi0 = InterpolatedUnivariateSpline(
    mass_data_rho_pi_plus_pi_minus_pi0_pi0,
    R_data_rho_pi_plus_pi_minus_pi0_pi0,
    ext=1
)

spline_omega_pi_plus_pi_minus_pi0 = InterpolatedUnivariateSpline(
    mass_data_omega_pi_plus_pi_minus_pi0,
    R_data_omega_pi_plus_pi_minus_pi0,
    ext=1
)

spline_phi_pi_plus_pi_minus_pi0 = InterpolatedUnivariateSpline(
    mass_data_phi_pi_plus_pi_minus_pi0,
    R_data_phi_pi_plus_pi_minus_pi0,
    ext=1
)

spline_omega_pi0_gamma = InterpolatedUnivariateSpline(
    mass_data_omega_pi0_gamma,
    R_data_omega_pi0_gamma,
    ext=1
)

spline_omega_phi_pi_plus_pi_minus_pi0 = InterpolatedUnivariateSpline(
    mass_data_omega_phi_pi_plus_pi_minus_pi0,
    R_data_omega_phi_pi_plus_pi_minus_pi0,
    ext=1
)

spline_phi_K_K = InterpolatedUnivariateSpline(
    mass_data_phi_K_K,
    R_data_phi_K_K,
    ext=1
)

spline_phi_K_K_pi = InterpolatedUnivariateSpline(
    mass_data_phi_K_K_pi,
    R_data_phi_K_K_pi,
    ext=1
)

spline_omega_other = InterpolatedUnivariateSpline(
    mass_data_omega_other,
    R_data_omega_other,
    ext=1
)

spline_phi_other = InterpolatedUnivariateSpline(
    mass_data_phi_other,
    R_data_phi_other,
    ext=1
)

spline_rho_other = InterpolatedUnivariateSpline(
    mass_data_rho_other,
    R_data_rho_other,
    ext=1
)

# Decay Profile

In [ ]:
def decay_profile(mX,alphaX,x):
  '''
  mX: CONSTANT mass of X boson in GeV
  alphaX: CONSTANT fine structure constant
  x: NUMPY ARRAY of guage charges [x_u, x_d, x_s, x_e, x_mu, x_nu_e, x_nu_mu]
  return: dict
  '''

  fermion_decays = gamma_ff(mX,alphaX,x) #returns gamma(X->e e_bar), gamma(X->mu mu_bar) , gamma(X->nu_e nu_e_bar) , gamma(X->nu_mu nu_mu_bar) , gamma(X->nu_tau nu_tau_bar)
  hadron_decays = gamma_hadrons(mX,alphaX,x) #returns gamma(X->pi+pi-) , gamma(X->pi+pi-pi0) , gamma(X->gammapi0)

  gamma_tot = np.sum(np.append(fermion_decays,hadron_decays))

  BR_ee = fermion_decays[0] / gamma_tot
  BR_mumu = fermion_decays[1] / gamma_tot
  BR_nunu_e = fermion_decays[2] / gamma_tot
  BR_nunu_mu = fermion_decays[3] / gamma_tot
  BR_nunu_tau = fermion_decays[4] / gamma_tot
  BR_nunu = BR_nunu_e + BR_nunu_mu + BR_nunu_tau
  BR_pi_plus_pi_minus = hadron_decays[0] / gamma_tot
  BR_pi_plus_pi_minus_pi_plus_pi_minus = hadron_decays[1] / gamma_tot
  BR_pi_plus_pi_minus_pi0 = hadron_decays[2] / gamma_tot
  BR_pi_plus_pi_minus_pi0_pi0 = hadron_decays[3] / gamma_tot
  BR_pi0_gamma = hadron_decays[4] / gamma_tot
  BR_K_K = hadron_decays[5] / gamma_tot
  BR_K_K_pi = hadron_decays[6] / gamma_tot
  BR_rho_other = hadron_decays[7] / gamma_tot
  BR_phi_other = hadron_decays[8] / gamma_tot
  BR_omega_other = hadron_decays[9] / gamma_tot

  return {
      'gamma_tot':gamma_tot,
      'BR_ee':BR_ee,
      'BR_mumu':BR_mumu,
      'BR_nunu': BR_nunu,
      'BR_pi_plus_pi_minus':BR_pi_plus_pi_minus,
      'BR_pi_plus_pi_minus_pi_plus_pi_minus':BR_pi_plus_pi_minus_pi_plus_pi_minus,
      'BR_pi_plus_pi_minus_pi0':BR_pi_plus_pi_minus_pi0,
      'BR_pi_plus_pi_minus_pi0_pi0':BR_pi_plus_pi_minus_pi0_pi0,
      'BR_pi0_gamma':BR_pi0_gamma,
      'BR_K_K': BR_K_K,
      'BR_K_K_pi': BR_K_K_pi,
      'BR_rho_other': BR_rho_other,
      'BR_phi_other': BR_phi_other,
      'BR_omega_other': BR_omega_other
  }

In [ ]:
def gamma_ff(mX,alphaX,x):
  '''
  x: gauge charges [x_u, x_d, x_s, x_e, x_mu, x_nu_e, x_nu_mu, x_nu_tau]
  return: 5 element 1D numpy array ordered gamma of [e+e-,mu+mu-,e_neutrino,mu_neutrino]
  '''
  if mX < 2*me:
    return np.zeros(5)

  x_leptons = x[3:] #ignoring quarks
  Cf = np.array([1,1,1/2,1/2,1/2])

  assert(len(x_leptons) == len(Cf))


  mf = np.array([me,mmu,0,0,0])

  try:
    gammas = Cf * alphaX * x_leptons**2 * mX * (1 + 2*mf**2/mX**2) * np.sqrt(1- 4*mf**2/mX**2) / 3
    assert not np.isnan(gammas).any()
  except AssertionError:
    gammas[1] = 0

  return gammas

In [ ]:
## eq 2.17 Ilten ##
def gamma_hadrons(mX,alphaX,x):
  '''
  Computes hadronic partial widths for a single mediator mass and coupling

  mX: float
    Mediator mass in GeV, not vectorized

  alphaX: float
    New force coupling strength, not vectorized

  x: numpy array
    gauge charges [x_u, x_d, x_s, x_e, x_mu, x_nu_e, x_nu_mu]

  return: 1D numpy array of length 10, containing partial decays into 10 hadronic states
  '''
  QX = np.diag([x[0] , x[1] , x[2]] , 0)

  return alphaX*mX/3*(R_X_rho(mX,QX) + R_X_omega(mX,QX) + R_X_phi(mX,QX) + R_X_omega_minus_phi(mX)) #this entire line might be garbage

# eq 2.18 Ilten ##
def R_X_rho(mX,QX):
  T_rho = 1/2 * np.diag([1,-1,0],0)
  return (2*np.trace(T_rho*QX))**2 * R_mu_rho(mX)

def R_X_omega(mX,QX):
  T_omega = 1/2 * np.diag([1,1,0],0)
  return (6*np.trace(T_omega*QX))**2 * R_mu_omega(mX)

def  R_X_phi(mX,QX):
  T_phi = 1/np.sqrt(2) * np.diag([0,0,1],0)
  return (3*np.sqrt(2)*np.trace(T_phi*QX))**2 * R_mu_phi(mX)

## Darkcast_vmd data retrieval ##
def R_X_omega_minus_phi(mX):
  arr = np.zeros(10)
  arr[2] = float(spline_omega_phi_pi_plus_pi_minus_pi0(mX))
  return arr

def R_mu_rho(mX):
  arr = np.zeros(10)
  arr[0] = float(spline_rho_pi_plus_pi_minus(mX))
  arr[3] = float(spline_rho_pi_plus_pi_minus_pi0_pi0(mX))
  arr[1] = float(spline_rho_pi_plus_pi_minus_pi_plus_pi_minus(mX))
  arr[7] = float(spline_rho_other(mX))
  return arr

def R_mu_omega(mX):
  arr = np.zeros(10)
  arr[4] = float(spline_omega_pi0_gamma(mX))
  arr[2] = float(spline_omega_pi_plus_pi_minus_pi0(mX))
  arr[9] = float(spline_omega_other(mX))

  return arr

def R_mu_phi(mX):
  arr = np.zeros(10)
  arr[2] = float(spline_phi_pi_plus_pi_minus_pi0(mX))
  arr[5] = float(spline_phi_K_K(mX))
  arr[6] = float(spline_phi_K_K_pi(mX))
  arr[8] = float(spline_phi_other(mX))

  return arr


# Branching Ratios

In [ ]:
def plot_branching_ratios(alphaX,x,title):
    '''
    Plots total hadron and lepton pair BRs on mX = (0,2) GeVs
    '''
    num = 1000
    mass_range = np.linspace(0,2,num)
    BR = np.array([decay_profile(m,alphaX,np.array(x)) for m in mass_range])

    BR_ee_range = np.zeros(num)
    BR_mumu_range = np.zeros(num)
    BR_nunu_range = np.zeros(num)
    BR_pi0_gamma_range = np.zeros(num)
    BR_pi_plus_pi_minus_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi0_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi0_pi0_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi_plus_pi_minus_range = np.zeros(num)
    BR_K_K_range = np.zeros(num)
    BR_K_K_pi_range = np.zeros(num)
    BR_omega_other_range = np.zeros(num)
    BR_phi_other_range = np.zeros(num)
    BR_rho_other_range = np.zeros(num)

    for i in range(num):
        BR_ee_range[i] = BR[i].get('BR_ee')
        BR_mumu_range[i] = BR[i].get('BR_mumu')
        BR_nunu_range[i] = BR[i].get('BR_nunu')
        BR_pi0_gamma_range[i] = BR[i].get('BR_pi0_gamma')
        BR_pi_plus_pi_minus_range[i] = BR[i].get('BR_pi_plus_pi_minus')
        BR_pi_plus_pi_minus_pi0_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi0')
        BR_pi_plus_pi_minus_pi0_pi0_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi0_pi0')
        BR_pi_plus_pi_minus_pi_plus_pi_minus_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi_plus_pi_minus')
        BR_K_K_range[i] = BR[i].get('BR_K_K')
        BR_K_K_pi_range[i] = BR[i].get('BR_K_K_pi')
        BR_omega_other_range[i] = BR[i].get('BR_omega_other')
        BR_phi_other_range[i] = BR[i].get('BR_phi_other')
        BR_rho_other_range[i] = BR[i].get('BR_rho_other')

    BR_hadron_range = np.sum(np.array([BR_pi0_gamma_range,
                                       BR_pi_plus_pi_minus_range,
                                       BR_pi_plus_pi_minus_pi0_range,
                                       BR_pi_plus_pi_minus_pi0_pi0_range,
                                       BR_pi_plus_pi_minus_pi_plus_pi_minus_range,
                                       BR_K_K_range,
                                       BR_K_K_pi_range,
                                       BR_omega_other_range,
                                       BR_phi_other_range,
                                       BR_rho_other_range]),
                             axis=0)

    if any(BR_hadron_range[~np.isnan(BR_hadron_range)] != 0):
      plt.plot(mass_range,BR_hadron_range,label='hadrons')

    if any(BR_ee_range[~np.isnan(BR_ee_range)] != 0):
      plt.plot(mass_range, BR_ee_range,label='e+e-')

    if any(BR_mumu_range[~np.isnan(BR_mumu_range)] != 0):
      plt.plot(mass_range,BR_mumu_range,label='mu+mu-')

    if any(BR_nunu_range[~np.isnan(BR_nunu_range)] != 0):
      plt.plot(mass_range,BR_nunu_range,label=f'v{'v'+'\u0305'}')

    plt.xlabel('mX')
    plt.ylabel('Branching Ratios')
    plt.legend()
    plt.title(str(title))
    plt.show()

In [ ]:
def plot_branching_ratios_individual(alphaX,x,title):
    '''
    Plots indidvual hadronic state and lepton pair BRs on mX = (0,2) GeVs
    '''
    num = 1000
    mass_range = np.linspace(0,2,num)
    BR = np.array([decay_profile(m,alphaX,np.array(x)) for m in mass_range])

    BR_ee_range = np.zeros(num)
    BR_mumu_range = np.zeros(num)
    BR_nu_range = np.zeros(num)
    BR_pi0_gamma_range = np.zeros(num)
    BR_pi_plus_pi_minus_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi0_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi0_pi0_range = np.zeros(num)
    BR_pi_plus_pi_minus_pi_plus_pi_minus_range = np.zeros(num)
    BR_K_K_range = np.zeros(num)
    BR_K_K_pi_range = np.zeros(num)
    BR_omega_other_range = np.zeros(num)
    BR_phi_other_range = np.zeros(num)
    BR_rho_other_range = np.zeros(num)

    for i in range(num):
        BR_ee_range[i] = BR[i].get('BR_ee')
        BR_mumu_range[i] = BR[i].get('BR_mumu')
        BR_nu_range[i] = BR[i].get('BR_nu')
        BR_pi0_gamma_range[i] = BR[i].get('BR_pi0_gamma')
        BR_pi_plus_pi_minus_range[i] = BR[i].get('BR_pi_plus_pi_minus')
        BR_pi_plus_pi_minus_pi0_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi0')
        BR_pi_plus_pi_minus_pi0_pi0_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi0_pi0')
        BR_pi_plus_pi_minus_pi_plus_pi_minus_range[i] = BR[i].get('BR_pi_plus_pi_minus_pi_plus_pi_minus')
        BR_K_K_range[i] = BR[i].get('BR_K_K')
        BR_K_K_pi_range[i] = BR[i].get('BR_K_K_pi')
        BR_omega_other_range[i] = BR[i].get('BR_omega_other')
        BR_phi_other_range[i] = BR[i].get('BR_phi_other')
        BR_rho_other_range[i] = BR[i].get('BR_rho_other')

    plt.figure(figsize=(12,6))

    plt.plot(mass_range, BR_ee_range,label='BR e+e-')
    plt.plot(mass_range,BR_mumu_range,label='BR mu+mu-')
    plt.plot(mass_range,BR_pi0_gamma_range,label='BR pi0 gamma')
    plt.plot(mass_range,BR_pi_plus_pi_minus_range,label='BR pi+pi-')
    plt.plot(mass_range,BR_pi_plus_pi_minus_pi0_range,label='BR pi+pi-pi0')
    plt.plot(mass_range,BR_pi_plus_pi_minus_pi0_pi0_range,label='BR pi+pi-pi0pi0')
    plt.plot(mass_range,BR_pi_plus_pi_minus_pi_plus_pi_minus_range,label='BR pi+pi-pi+pi-')
    plt.plot(mass_range,BR_K_K_range,label='BR KK')
    plt.plot(mass_range, BR_K_K_pi_range, label='BR KK pi')
    plt.plot(mass_range, BR_omega_other_range, label='BR omega other')
    plt.plot(mass_range, BR_phi_other_range, label='BR phi other')
    plt.plot(mass_range, BR_rho_other_range, label='BR rho other')

    plt.xlabel('mX')
    plt.ylabel('Branching Ratios')
    plt.title(str(title))
    plt.legend()
    plt.yscale('log')
    plt.ylim(1e-3,None)
    plt.show()

# Partial Widths

In [ ]:
def plot_partial_widths(x,alphaX):
  mass_range = np.linspace(0,2,1000)
  gamma_fermions_list = [gamma_ff(m,alphaX,np.array(x)) for m in mass_range]
  gamma_hadrons_list = [gamma_hadrons(m,alphaX,np.array(x)) for m in mass_range]

  gamma_mumu_range = np.zeros(1000)
  gamma_ee_range = np.zeros(1000)
  gamma_nunu_range = np.zeros(1000)

  gamma_pi0_gamma_range = np.zeros(1000)
  gamma_pi_plus_pi_minus_range = np.zeros(1000)
  gamma_pi_plus_pi_minus_pi0_range = np.zeros(1000)
  gamma_pi_plus_pi_minus_pi0_pi0_range = np.zeros(1000)
  gamma_pi_plus_pi_minus_pi_plus_pi_minus_range = np.zeros(1000)

  gamma_K_K_range = np.zeros(1000)
  gamma_K_K_pi_range = np.zeros(1000)
  gamma_omega_other_range = np.zeros(1000)
  gamma_phi_other_range = np.zeros(1000)
  gamma_rho_other_range = np.zeros(1000)

  for i in range(1000):
    gamma_ee_range[i] = gamma_fermions_list[i][0]
    gamma_mumu_range[i] = gamma_fermions_list[i][1]
    gamma_nunu_range[i] = gamma_fermions_list[i][2] + gamma_fermions_list[i][3] + gamma_fermions_list[i][4]

    gamma_pi0_gamma_range[i] = gamma_hadrons_list[i][4]
    gamma_pi_plus_pi_minus_range[i] = gamma_hadrons_list[i][0]
    gamma_pi_plus_pi_minus_pi0_range[i] = gamma_hadrons_list[i][2]
    gamma_pi_plus_pi_minus_pi0_pi0_range[i] = gamma_hadrons_list[i][3]
    gamma_pi_plus_pi_minus_pi_plus_pi_minus_range[i] = gamma_hadrons_list[i][1]

    gamma_K_K_range[i] = gamma_hadrons_list[i][5]
    gamma_K_K_pi_range[i] = gamma_hadrons_list[i][6]
    gamma_omega_other_range[i] = gamma_hadrons_list[i][9]
    gamma_phi_other_range[i] = gamma_hadrons_list[i][8]
    gamma_rho_other_range[i] = gamma_hadrons_list[i][7]


  plt.figure(figsize=(12,6))
  plt.plot(mass_range,gamma_ee_range,label='gamma_ee')
  plt.plot(mass_range,gamma_mumu_range,label='gamma_mumu')
  plt.plot(mass_range,gamma_pi0_gamma_range,label='gamma_pi0_gamma')
  plt.plot(mass_range,gamma_pi_plus_pi_minus_range,label='gamma_pi_plus_pi_minus')
  plt.plot(mass_range,gamma_pi_plus_pi_minus_pi0_range,label='gamma_pi_plus_pi_minus_pi0')
  plt.plot(mass_range,gamma_pi_plus_pi_minus_pi0_pi0_range,label='gamma_pi_plus_pi_minus_pi0_pi0')
  plt.plot(mass_range,gamma_pi_plus_pi_minus_pi_plus_pi_minus_range,label='gamma_pi_plus_pi_minus_pi_plus_pi_minus')
  plt.plot(mass_range, gamma_K_K_range, label='gamma_K_K')
  plt.plot(mass_range, gamma_K_K_pi_range, label='gamma_K_K_pi')
  plt.plot(mass_range, gamma_omega_other_range, label='gamma_omega_other')
  plt.plot(mass_range, gamma_phi_other_range, label='gamma_phi_other')
  plt.plot(mass_range, gamma_rho_other_range, label='gamma_rho_other')
  plt.plot(mass_range,gamma_nunu_range, label='gama_nunu')


  plt.legend()
  plt.xlabel('mX')
  plt.ylabel('partial decay width')
  plt.yscale('log')
  plt.ylim(10e-15,None)
  plt.show()


# Decay Length

In [ ]:
def plot_decay_length(alphaX,x,title):
    '''
    Plots proper decay length for mX on (0,2) GeVs
    '''
    num = 1000
    mass_range = np.linspace(0,2,num)
    BR = np.array([decay_profile(m,alphaX,np.array(x)) for m in mass_range])
    gamma_tot_range = np.zeros(num)
    for i in range(1000):
        gamma_tot_range[i] = BR[i].get('gamma_tot')

    lifetime_range = 1/gamma_tot_range

    L_range = lifetime_range * 0.197 # fm

    plt.plot(mass_range,L_range)
    plt.yscale('log')
    plt.ylabel('decay length (L) [fm]')
    plt.xlabel('mX')
    plt.title(str(title))
    plt.show()

In [ ]:
def get_decay_length(mX,alphaX,x):
  '''
  returns: float, proper decay length
  '''
  gamma_tot = decay_profile(mX,alphaX,x).get('gamma_tot')
  return 0.197 / gamma_tot #units?

#Notes

In [ ]:
'''
Hadronic state array
0: pi+ pi-
1: pi+ pi- pi+ pi-
2: pi+ pi- pi0
3: pi+ pi- pi0 pi0
4: pi0 gamma
5: K K
6: K K pi
7: rho other
8: phi other
9: omega other

Citations for formula
gamma_ff Serendipity eq2.13, Chargephobic eq25
gamma_hadrons Serendipity eq2.17
R_X_V_hadrons Serendipity eq2.18

darkast data
https://gitlab.com/darkcast/releases/-/tree/master/darkcast/data
'''